<a href="https://colab.research.google.com/github/NiyatiPaghda/brain-tumor-mri-study/blob/main/conv_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1) Imports all the necessary libraries. We need TensorFlow/Keras to build our neural network, OpenCV (cv2) and NumPy to process our images as array, and pickle to save our processed data so we don't have to reload and resize thousands of images every single time we run the notebook. Also includes the code to mount Google Drive b/c dataset is stored there.

In [1]:
# Import necessary libraries for deep learning and image processing
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Conv2D, MaxPooling2D
import pickle
import os
import numpy as np
import cv2
import random

# Uncomment following first time running
# from google.colab import drive
# drive.mount('/content/drive/')

Mounted at /content/drive/


2. Setting up file path, define the categories (Dogs and Cats), and set our target image size to 50 x 50 pixels. We also write a function that loops through the folders, reads every image in grayscale, resizes it, and adds it to our training_data list.

In [2]:
# Define path to dataset and categories
DATADIR = "/content/drive/MyDrive/Independent Study/PetImages"
CATEGORIES = ["Dog", "Cat"]

# Set uniform image size for the neural network input
IMG_SIZE = 50

training_data = []

def create_training_data():
    for category in CATEGORIES:
        path = os.path.join(DATADIR, category) # Path to cats and dogs directory
        class_num = CATEGORIES.index(category) # Get label: 0 for Dog, 1 for Cat (or vice versa)

        for img in os.listdir(path):
            try:
              # Read image in grayscale to reduce complexity and file size
              img_array = cv2.imread(os.path.join(path,img), cv2.IMREAD_GRAYSCALE)
              # Resize image to our standard 50x50 pixels
              new_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))
              # Add the image array and its label to our training data list
              training_data.append([new_array, class_num])
            except Exception as e:
              # Catch broken or corrupted images so the loop doesn't crash
              pass

# Run the function to build our dataset
create_training_data()

# Shuffle the data so the model doesn't see all dogs first and all cats second
random.shuffle(training_data)

3. Neural networks need inputs (X) and labels (y) sperated. We loop through our shuffled training_data, pull out the image array into X, and pull out the category numbers into y.

In [3]:
X = []
y = []

# Separate features (images) and labels (categories)
for features, label in training_data:
    X.append(features)
    y.append(label)

4. Processing all those images takes a long time. To save time on future runs, we use Pickle to save our X and y lists directly onto our hard drive as binary files (X.pickle and y.pickle).

In [4]:
# Save the feature list (X) to a pickle file
with open("X.pickle", "wb") as f:
    pickle.dump(X, f)

# Save the label list (y) to a pickle file
with open("y.pickle", "wb") as f:
    pickle.dump(y, f)

5. Now we load our data back from the pickle files, convert them into NumPy arrays, and reshape X so Tensorflow knows the dimensions. We also normalize our image pixel values by dividing by 255.0. This scales pixel colors from a range of 0-255 down to 0-1, which helps the neural network learn much faster and more accurately.

In [5]:
# Load the data back from pickle files and convert to NumPy arrays
X = np.asarray(pickle.load(open("X.pickle", "rb")))
y = np.asarray(pickle.load(open("y.pickle", "rb")))

# Reshape X to match what the CNN expects: (samples, height, width, color_channels)
# Here, color_channels = 1 because our images are grayscale
X = X.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

# Normalize pixel values from 0-255 to 0.0-1.0
X = X / 255.0

6. This is where the magic happens...
- Conv2D & MaxPoooling2D: These layers scan the image to extract features like edges, shapes, and textures (convolution), and then downsize the image to keep computations manageable (max pooing).
- Flatten: COnverst our 2D feature maps into a 1D vector so we can feed it into standard dense layers.
- Dense & Activations (Sigmoid): The final output layer uses a sigmoid activation function to output a probability between 0 and 1 (deciding if it's a dog or a cat).

In [6]:
model = Sequential()

# -- First Convolutional Block --
model.add(Conv2D(64, (3,3), input_shape = X.shape[1:]))
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2,2)))

# -- Second Convolutional Block --
model.add(Conv2D(64, (3,3)))
model.add(Activation("relu"))
model.add(MaxPooling2D(pool_size=(2,2)))

# -- Fully Connected / Dense Layers --
model.add(Flatten())
model.add(Dense(64))
model.add(Activation("relu"))

# -- Output Layer --
model.add(Dense(1))
model.add(Activation('sigmoid')) #Sigmoid outputs 0 or 1 for binary classifications


/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


7. Finally, configuring how the model learns by compiling it with an optimizer (adam) and a loss function (binary_crossentropy). Then, we call model.fit() to train the neural network on our data for 3 epochs, setting aside 10% of the data for validations to check how well it performs on unseen images.

In [ ]:
# Compile the model
model.compile(loss="binary_crossentropy",
              optimizer="adam",
              metrics=['accuracy'])

# Train the model
model.fit(X, y, batch_size=32, epochs=10, validation_split=0.1)

Epoch 1/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 8s 8ms/step - accuracy: 0.9924 - loss: 0.0231 - val_accuracy: 0.7844 - val_loss: 1.5963
Epoch 2/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9934 - loss: 0.0203 - val_accuracy: 0.7620 - val_loss: 1.6736
Epoch 3/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9904 - loss: 0.0309 - val_accuracy: 0.7840 - val_loss: 1.5589
Epoch 4/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.9982 - loss: 0.0062 - val_accuracy: 0.7820 - val_loss: 1.5891
Epoch 5/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9973 - loss: 0.0093 - val_accuracy: 0.7836 - val_loss: 1.7208
Epoch 6/10
704/704 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.9905 - loss: 0.0281 - val_accuracy: 0.7848 - val_loss: 1.5520
Epoch 7/10
 23/704 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9811 - loss: 0.0636